In [9]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import numpy as np
import pandas as pd

from src.evaluation import load_train, cross_validate, calibration_by_exposure, exposure_band_reference, PROCESSED
from src.gbm import fit_predict

train = load_train()

def baseline(tr, va):
    return (va["Exposure"] * tr["ClaimNb"].sum() / tr["Exposure"].sum()).to_numpy()

oof_g, s_g = cross_validate(train, fit_predict)          # same loop, folds and metric as every other model
_, s_b = cross_validate(train, baseline)
_, s_e = cross_validate(train, exposure_band_reference)

print("GBM v1      :", s_g.round(4), "| mean", round(s_g.mean(), 4), "+/-", round(s_g.std(), 4))
print("vs baseline :", (s_b - s_g).round(4), "| mean", round((s_b - s_g).mean(), 4))
print("vs yardstick:", (s_e - s_g).round(4), "| mean", round((s_e - s_g).mean(), 4))
print("\noverall actual/predicted:", round(train["ClaimNb"].sum() / oof_g.sum(), 4), "\n")
print(calibration_by_exposure(train, oof_g))

# Overwrites the earlier file, which was produced WITHOUT the bm_off_ladder feature
pd.DataFrame({"IDpol": train["IDpol"], "pred_claims": oof_g}).to_csv(PROCESSED / "oof_gbm.csv", index=False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
GBM v1      : [0.302  0.3038 0.3026 0.3095 0.3011] | mean 0.3038 +/- 0.003
vs baseline : [0.0271 0.0267 0.0304 0.0254 0.0274] | mean 0.0274
vs yardstick: [0.0145 0.0142 0.0173 0.0121 0.0142] | mean 0.0145

overall actual/predicted: 1.0005 

             policies  actual  predicted  actual/predicted
Exposure                                                  
(0.0, 0.1]     102054    2785      841.7             3.309
(0.1, 0.25]     76054    2890     1750.9             1.651
(0.25, 0.5]    105108    5212     4896.8             1.064
(0.5, 0.75]     73967    4822     5390.1             0.895
(0.75, 1.0]    184249   13123    15938.9             0.823


# Gradient boosting with Poisson loss: claim frequency

**Goal:** a flexible model of the expected number of claims per policy, to be compared (same folds, same metric) with the Poisson GLM built in parallel.

**Gradient boosting in one paragraph.** The model is a sum of many small decision trees built one after the other. Each new tree is fitted to the *errors* of the trees so far, and its contribution is shrunk by a small `learning_rate`. With the Poisson objective the trees add up to `log(frequency)`, exactly like the GLM's linear predictor, but each tree can split on any feature and combine features, so non-linear effects and interactions are found automatically.

**Exposure.** The target is the frequency `ClaimNb / Exposure` and `Exposure` is the **sample weight**. This is equivalent to modelling claim counts with an exposure offset. The model predicts a frequency, and `expected claims = prediction x exposure`.

**Rules (same as the baseline and the GLM)**
* Training set only; the test set stays locked until both models are final.
* Grouped 5-fold CV with the frozen folds, scored by the Poisson deviance (lower is better).
* Features: the raw columns (trees need no bands, logs or scaling), plus one derived flag, `bm_off_ladder` (Step 4). Never `Area`, `Exposure`, IDs or target-side columns.
* Everything that learns from data (the trees, the category level lists) is fitted on a training part only.

**Plan:** 1. one fold and a learning curve, 2. learning curves on all folds, 3. a small tuning, 4. one representation experiment, 5. final run through the shared module `src/gbm.py`.

In [1]:
# Cell 1: setup, training data and the reference baseline
# (Magics must stay alone on their line: no comments after them.)
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")      # lets the notebook import our project code from src/
import numpy as np
import pandas as pd
import lightgbm as lgb

from src.evaluation import (load_train, poisson_deviance, cross_validate, calibration_by_exposure,
                            exposure_band_reference, N_FOLDS, PROCESSED)
from src.features import LADDER          # the bonus-malus ladder, used for the bm_off_ladder flag (Step 4)

train = load_train()
print(train.shape)                       # your run: (541432, 23)

def baseline(tr, va):
    """The reference: one constant rate learned on the training part, scaled by exposure."""
    return (va["Exposure"] * tr["ClaimNb"].sum() / tr["Exposure"].sum()).to_numpy()

(541432, 23)


## Raw features for the trees
A split such as `BonusMalus <= 62` does not change under any monotone transformation, so the trees get the **raw** numeric columns. Categories use the pandas `category` dtype, which LightGBM splits on natively (no one-hot). The level lists come from the *fitting* part only (names, no target information).

In [2]:
# Cell 2: feature lists, the matrix builder, and shared helpers
NUM_RAW = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]
CAT_RAW = ["VehBrand", "VehGas", "Region"]
FEATURES = NUM_RAW + CAT_RAW             # Step 4 will test adding "bm_off_ladder"; Area is excluded (binned Density)

def make_X(d, levels, features):
    """Model matrix for the trees. `levels`: category level names learned on the FITTING part."""
    X = d[[f for f in features if f != "bm_off_ladder"]].copy()
    if "bm_off_ladder" in features:
        X["bm_off_ladder"] = (~d["BonusMalus"].isin(LADDER)).astype(int)     # not a claim-free rung: likely a recent claim
    for c in CAT_RAW:
        if c in features:
            X[c] = pd.Categorical(X[c], categories=levels[c])
    return X[features]

# Starting hyperparameters (tuned in Step 3):
#   learning_rate     : size of each tree's contribution (smaller = needs more trees, usually better)
#   num_leaves        : complexity of each tree (16 leaves allows interactions between about 4 features)
#   min_child_samples : minimum rows in a leaf (500 rows is only about 25 claims at a 5% claim rate)
#   subsample, colsample_bytree : random share of rows / columns per tree, a defense against overfitting
#   reg_lambda        : L2 penalty on the leaf values
PARAMS = dict(objective="poisson", learning_rate=0.05, n_estimators=500, num_leaves=16, min_child_samples=500,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=42, verbose=-1)

def expected(model, d, X, k=None):
    """Expected claims = predicted frequency x exposure, using only the first k trees (None = all)."""
    return model.predict(X, num_iteration=k) * d["Exposure"].to_numpy()

masks = [(train["Fold"] == f).to_numpy() for f in range(N_FOLDS)]     # boolean row masks of the 5 folds

## Step 1: one fold, one model, the learning curve
Fold 0 is the validation part, the other four folds are the training part. We fit 500 trees and then read the predictions **after the first k trees** (`num_iteration=k`), which gives the whole learning curve from one fit.

* **Train deviance** keeps falling: the model keeps fitting the training data.
* **Validation deviance** falls, flattens and may rise: the lowest point is the best number of trees, and the growing gap between the two curves is the overfitting. The *sign* of the gap is not meaningful (fold 0 is an easier fold than average); only how it changes is.

In [3]:
# Cell 3: fit on fold 0 and read the learning curve
tr, va = train[train["Fold"] != 0], train[train["Fold"] == 0]
levels = {c: sorted(tr[c].unique()) for c in CAT_RAW}                  # level names from the training part only
X_tr, X_va = make_X(tr, levels, FEATURES), make_X(va, levels, FEATURES)

model = lgb.LGBMRegressor(**PARAMS)
model.fit(X_tr, tr["ClaimNb"] / tr["Exposure"], sample_weight=tr["Exposure"])     # frequency target, exposure weight

rows = []
for k in [25, 50, 100, 200, 300, 400, 500]:
    rows.append((k, poisson_deviance(tr["ClaimNb"], expected(model, tr, X_tr, k)),
                    poisson_deviance(va["ClaimNb"], expected(model, va, X_va, k))))
curve = pd.DataFrame(rows, columns=["trees", "train", "validation"]).round(4)
print(curve.to_string(index=False))

d_base = poisson_deviance(va["ClaimNb"], baseline(tr, va))
assert np.isclose(d_base, 0.3291, atol=1e-4)               # fold-0 baseline from the reference run
best = curve.loc[curve["validation"].idxmin()]
print("baseline on fold 0:", round(d_base, 4), "| best validation:", best["validation"], "at", int(best["trees"]), "trees")

# Your run:  trees  train   validation
#              25   0.3140  0.3124
#              50   0.3094  0.3083
#             100   0.3055  0.3052
#             200   0.3026  0.3036
#             300   0.3008  0.3031
#             400   0.2995  0.3028
#             500   0.2985  0.3027      -> 0.0264 (8.0%) better than the baseline 0.3291, still flattening at 500 trees

 trees  train  validation
    25 0.3140      0.3124
    50 0.3094      0.3083
   100 0.3055      0.3052
   200 0.3026      0.3036
   300 0.3008      0.3031
   400 0.2995      0.3028
   500 0.2985      0.3027
baseline on fold 0: 0.3291 | best validation: 0.3027 at 500 trees


In [4]:
# Cell 4: what does the model rely on? (gain importance, in % of the total)
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=FEATURES)
print((100 * imp / imp.sum()).round(1).sort_values(ascending=False).to_string())

# Your run: BonusMalus 27.6, VehAge 26.8, VehBrand 10.4, DrivAge 9.6, Region 7.9, VehPower 7.7, Density 5.3, VehGas 4.6
# The top two match the EDA. CAUTION: gain importance is measured on TRAINING data and favours features with many levels,
# so VehBrand / Region / VehPower may be inflated by noise or by interactions (e.g. brand B12 x new vehicles).
# A permutation test on validation data would be the stricter check.

BonusMalus    27.6
VehAge        26.8
VehBrand      10.4
DrivAge        9.6
Region         7.9
VehPower       7.7
Density        5.3
VehGas         4.6


## Step 2: the learning curve on all five folds
One fit per fold with 1,000 trees; the predictions after 100, 200, ... trees give the curve for every fold. Choosing the tree count from all folds is safer than from fold 0 alone. `cv_gbm` is reused by Steps 3 and 4.

In [ ]:
# Cell 5: learning curves on all folds, paired comparison, calibration
N_TREES = 1000
CHECKPOINTS = [100, 200, 300, 500, 700, 1000]

def cv_gbm(params, checkpoints, features):
    """Grouped 5-fold CV with ONE fit per fold of max(checkpoints) trees.
    Returns scores (rows = checkpoints, columns = folds) and oof (policies x checkpoints) of expected claims."""
    n_trees = max(checkpoints)
    scores = np.zeros((len(checkpoints), N_FOLDS))
    oof = np.zeros((len(train), len(checkpoints)))
    for f in range(N_FOLDS):
        tr_f, va_f = train[~masks[f]], train[masks[f]]
        lv = {c: sorted(tr_f[c].unique()) for c in CAT_RAW}              # levels from the fitting part only
        m = lgb.LGBMRegressor(**{**params, "n_estimators": n_trees})
        m.fit(make_X(tr_f, lv, features), tr_f["ClaimNb"] / tr_f["Exposure"], sample_weight=tr_f["Exposure"])
        X_v = make_X(va_f, lv, features)
        for j, k in enumerate(checkpoints):
            oof[masks[f], j] = expected(m, va_f, X_v, k)
            scores[j, f] = poisson_deviance(va_f["ClaimNb"], oof[masks[f], j])
    return scores, oof

scores, oof = cv_gbm(PARAMS, CHECKPOINTS, FEATURES)
table = pd.DataFrame(scores, index=CHECKPOINTS, columns=[f"fold {f}" for f in range(N_FOLDS)])
table["mean"] = scores.mean(axis=1)
print(table.round(4).to_string())
best = int(np.argmin(scores.mean(axis=1)))
print("\nbest checkpoint:", CHECKPOINTS[best], "trees")

_, s_b = cross_validate(train, baseline)
_, s_e = cross_validate(train, exposure_band_reference)
s_g = scores[best]
print("improvement vs baseline  per fold:", (s_b - s_g).round(4), "| mean", round((s_b - s_g).mean(), 4))
print("improvement vs yardstick per fold:", (s_e - s_g).round(4), "| mean", round((s_e - s_g).mean(), 4))
print("\n", calibration_by_exposure(train, oof[:, best]))

# Your run (mean deviance by trees): 100: 0.3067 | 200: 0.3052 | 300: 0.3048 | 500: 0.3045 | 700: 0.3044 | 1000: 0.3044
# -> converged. 700 and 1000 tie at 4 decimals, so we keep 700 (the simplest model within noise).
# Improvement vs baseline: 0.0266, 0.0261, 0.0301, 0.0246, 0.0267 (mean 0.0268, 8.1%) -> positive in EVERY fold: real.
# Improvement vs yardstick: mean 0.0139 -> about twice what exposure alone provides.
# Calibration (actual/predicted) baseline -> boosting: <=0.1: 4.58 -> 3.29 | 0.1-0.25: 2.06 -> 1.65 | 0.25-0.5: 1.26 -> 1.06
#                                                      0.5-0.75: 1.02 -> 0.90 | 0.75-1.0: 0.73 -> 0.83
# Reading: partly improved, NOT fixed: short policies are still under-predicted, full-year policies over-predicted by ~21%.
# The model never sees exposure, so no feature can fully fix this: a known limitation shared with the GLM.

## Step 3: a small tuning
One setting at a time around the current configuration, with `learning_rate=0.1` and at most 300 trees to keep it fast. A change is **adopted only if it is better in at least 4 of 5 folds and the mean gain is at least about 0.0005**: the paired gap between two reference models varied by only about 0.0004 across folds, so smaller differences cannot be told apart from noise. (This one-at-a-time search does not explore combinations of settings.)

In [ ]:
# Cell 6: one-at-a-time tuning (several minutes: 8 configurations x 5 folds x 300 trees)
CHECKPOINTS_T = [100, 200, 300]
BASE = {**PARAMS, "learning_rate": 0.1}                    # faster search: bigger steps, fewer trees

CONFIGS = {
    "base": {},
    "leaves 8": {"num_leaves": 8},                         # simpler trees
    "leaves 32": {"num_leaves": 32},                       # more complex trees
    "min_child 200": {"min_child_samples": 200},           # smaller leaves allowed
    "min_child 1500": {"min_child_samples": 1500},         # larger leaves required
    "lambda 10": {"reg_lambda": 10.0},                     # stronger shrinkage of the leaf values
    "colsample 0.5": {"colsample_bytree": 0.5},            # fewer columns per tree
    "cat regularized": {"cat_smooth": 100, "min_data_per_group": 500},    # calmer splits on categories
}
results = {}
for name, change in CONFIGS.items():
    results[name], _ = cv_gbm({**BASE, **change}, CHECKPOINTS_T, FEATURES)
    print(name, "done")

def best_row(res):
    j = int(np.argmin(res.mean(axis=1)))
    return CHECKPOINTS_T[j], res[j]

k0, s0 = best_row(results["base"])
rows = []
for name, res in results.items():
    k, s = best_row(res)
    rows.append({"config": name, "trees": k, "mean_deviance": s.mean(), "gain_vs_base": (s0 - s).mean(),
                 "folds_better": int((s < s0).sum()), "worst_fold_gain": (s0 - s).min()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

# Your run: base 0.3046 | leaves 8: -0.0009 (0/5 folds) | leaves 32: +0.0002 (4/5) | min_child 200: +0.0001 (4/5)
#           min_child 1500: -0.0002 | lambda 10: 0.0000 | colsample 0.5: -0.0004 | cat regularized: +0.0001 (4/5)
# -> NO change qualifies (best gain 0.0002). Trees with 8 leaves lose in every fold (interactions matter),
#    but 32 leaves add almost nothing: the model is saturated. Hyperparameters are not the bottleneck.

base done
leaves 8 done
leaves 32 done
min_child 200 done
min_child 1500 done
lambda 10 done
colsample 0.5 done
cat regularized done
         config  trees  mean_deviance  gain_vs_base  folds_better  worst_fold_gain
           base    300         0.3046        0.0000             0           0.0000
       leaves 8    300         0.3055       -0.0009             0          -0.0011
      leaves 32    200         0.3044        0.0002             4          -0.0001
  min_child 200    300         0.3045        0.0001             4          -0.0000
 min_child 1500    300         0.3048       -0.0002             0          -0.0004
      lambda 10    300         0.3046        0.0000             2          -0.0002
  colsample 0.5    300         0.3050       -0.0004             0          -0.0005
cat regularized    300         0.3045        0.0001             4          -0.0000


## Step 4: one representation experiment
The trees see the raw bonus-malus values, but "off the ladder" (a value that is not a claim-free rung, which most likely marks a recent claim) is a union of many separate values that takes many splits to isolate. The flag `bm_off_ladder` expresses it in one split. It is derived from the same column, so it adds **no new information**, which keeps the comparison with the GLM (which already uses it) fair.

In [ ]:
# Cell 7: raw features vs raw features + bm_off_ladder (same settings, same folds, paired)
scores_raw, _ = cv_gbm(BASE, CHECKPOINTS_T, FEATURES)
scores_flag, _ = cv_gbm(BASE, CHECKPOINTS_T, FEATURES + ["bm_off_ladder"])
k0, s0 = best_row(scores_raw)
k1, s1 = best_row(scores_flag)
print("raw features   :", k0, "trees |", s0.round(4), "| mean", round(s0.mean(), 4))
print("+ bm_off_ladder:", k1, "trees |", s1.round(4), "| mean", round(s1.mean(), 4))
print("gain per fold:", (s0 - s1).round(4), "| mean", round((s0 - s1).mean(), 4), "| folds better:", int((s1 < s0).sum()))

# Your run: 0.3046 -> 0.3039; gain 0.0007, 0.0008, 0.0006, 0.0007, 0.0008 (5 of 5 folds, above the 0.0005 threshold).
# ADOPTED: three times the best hyperparameter change. Even with raw values available, the flag helps the trees, which
# also suggests the off-ladder signal is real (my interpretation of it as 'recent claim' remains a hypothesis).

raw features   : 300 trees | [0.3028 0.3045 0.3032 0.3103 0.3022] | mean 0.3046
+ bm_off_ladder: 300 trees | [0.3021 0.3037 0.3026 0.3096 0.3014] | mean 0.3039
gain per fold: [0.0007 0.0008 0.0006 0.0007 0.0008] | mean 0.0007 | folds better: 5


## Step 5: the final configuration, through the shared module
The settings above are saved in `src/gbm.py` (700 trees, learning rate 0.05, 16 leaves, features = the 8 raw columns + `bm_off_ladder`). Its `fit_predict(train_part, valid_part)` has the signature the shared `cross_validate` expects, so the boosting model is scored by exactly the same loop, folds and metric as the baseline and the GLM.

In [ ]:
# Cell 8: final run, paired comparison with the references, calibration, and the out-of-fold predictions for the comparison
from src.gbm import fit_predict, PARAMS as FINAL_PARAMS, FEATURES as FINAL_FEATURES
print(FINAL_FEATURES)
print(FINAL_PARAMS)

oof_g, s_g = cross_validate(train, fit_predict)          # also cross-checks the deviance against scikit-learn
_, s_b = cross_validate(train, baseline)
_, s_e = cross_validate(train, exposure_band_reference)
assert np.isclose(s_b.mean(), 0.3312, atol=1e-4)         # the shared baseline still reproduces the reference

print("GBM final   :", s_g.round(4), "| mean", round(s_g.mean(), 4), "+/-", round(s_g.std(), 4))
print("vs baseline :", (s_b - s_g).round(4), "| mean", round((s_b - s_g).mean(), 4))
print("vs yardstick:", (s_e - s_g).round(4), "| mean", round((s_e - s_g).mean(), 4))
print("\noverall actual/predicted:", round(train["ClaimNb"].sum() / oof_g.sum(), 4), "\n")
print(calibration_by_exposure(train, oof_g))

# Out-of-fold expected claims per policy: the input of the final comparison with the GLM (partner saves oof_glm.csv the same way)
pd.DataFrame({"IDpol": train["IDpol"], "pred_claims": oof_g}).to_csv(PROCESSED / "oof_gbm.csv", index=False)

# Expectation (hypothesis, not yet run): mean deviance about 0.3037 (0.0007 below the raw-feature 0.3044),
# gains over the baseline near 0.027 in every fold, and a calibration by exposure similar to Step 2.

['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'Density', 'VehBrand', 'VehGas', 'Region', 'bm_off_ladder']
{'objective': 'poisson', 'learning_rate': 0.05, 'n_estimators': 700, 'num_leaves': 16, 'min_child_samples': 500, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'random_state': 42, 'verbose': -1}
GBM final   : [0.302  0.3038 0.3026 0.3095 0.3011] | mean 0.3038 +/- 0.003
vs baseline : [0.0271 0.0267 0.0304 0.0254 0.0274] | mean 0.0274
vs yardstick: [0.0145 0.0142 0.0173 0.0121 0.0142] | mean 0.0145

overall actual/predicted: 1.0005 

             policies  actual  predicted  actual/predicted
Exposure                                                  
(0.0, 0.1]     102054    2785      841.7             3.309
(0.1, 0.25]     76054    2890     1750.9             1.651
(0.25, 0.5]    105108    5212     4896.8             1.064
(0.5, 0.75]     73967    4822     5390.1             0.895
(0.75, 1.0]    184249   13123    15938.9             0.823


## Summary

| Quantity | Value |
|---|---|
| Baseline (constant rate x exposure) | 0.3312 |
| Exposure-band yardstick (not a model) | 0.3183 |
| Boosting, raw features, 700 to 1000 trees | 0.3044 (0.0268 below the baseline, positive in all 5 folds) |
| Effect of `bm_off_ladder` | about 0.0007 (5 of 5 folds) |
| Best hyperparameter change | 0.0002 (below the noise threshold): none adopted |
| Calibration by exposure | partly improved, still under-predicts short policies and over-predicts full-year ones |
